In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [15]:
players = pd.read_csv("../data/processed/players_master.csv")
print(players.shape)
print(players.columns.tolist())

(1248, 25)
['broad_pos', 'caps', 'club', 'defending', 'dob', 'dribbling', 'fc26_club', 'fc26_long_name', 'fc26_match_method', 'fc26_overall', 'goals', 'match_type', 'name', 'overall', 'pace', 'passing', 'physic', 'player_positions', 'position', 'shooting', 'team', 'understat_mins', 'understat_name', 'understat_xA_per90', 'understat_xG_per90']


In [16]:
def weighted_avg(df, value_col, weight_col):
    return (df[value_col] * df[weight_col]).sum() / df[weight_col].sum()

In [22]:
rows = []
for team_name in players['team'].unique():
    team = players[players['team'] == team_name]
    
    ATT = team[team['broad_pos'] == 'ATT']
    MID = team[team['broad_pos'] == 'MID']
    DEF = team[team['broad_pos'] == 'DEF']
    GK  = team[team['broad_pos'] == 'GK']
    
    top11 = team.nlargest(11, 'overall')
    bench = team[~team.index.isin(top11.index)]

    att_und_rows = []
    for _, p in ATT.iterrows():
        if pd.notna(p.get('understat_xG_per90')):
            att_und_rows.append({
                'xG_per90': p['understat_xG_per90'],
                'xA_per90': p['understat_xA_per90'],
                'minutes' : p['understat_mins'],
            })

    att_coverage = len(att_und_rows) / len(ATT) if len(ATT) > 0 else 0

    if att_und_rows and att_coverage >= 0.4:
        att_und = pd.DataFrame(att_und_rows)
        atk_xG = (att_und['xG_per90'] * att_und['minutes']).sum() / att_und['minutes'].sum()
        atk_xA = (att_und['xA_per90'] * att_und['minutes']).sum() / att_und['minutes'].sum()
    else:
        atk_xG = None
        atk_xA = None

    rows.append({
        'team'          : team_name,
        'atk_overall'   : weighted_avg(ATT, 'overall',   'overall'),
        'atk_shooting'  : weighted_avg(ATT, 'shooting',  'overall'),
        'atk_pace'      : weighted_avg(ATT, 'pace',      'overall'),
        'atk_dribbling' : weighted_avg(ATT, 'dribbling', 'overall'),
        'mid_overall'   : weighted_avg(MID, 'overall',   'overall'),
        'mid_passing'   : weighted_avg(MID, 'passing',   'overall'),
        'mid_physic'    : weighted_avg(MID, 'physic',    'overall'),
        'def_overall'   : weighted_avg(DEF, 'overall',   'overall'),
        'def_defending' : weighted_avg(DEF, 'defending', 'overall'),
        'def_physic'    : weighted_avg(DEF, 'physic',    'overall'),
        'gk_overall'    : weighted_avg(GK,  'overall',   'overall'),
        'top11_overall' : top11['overall'].mean(),
        'squad_depth'   : top11['overall'].mean() - bench['overall'].mean(),
        'avg_caps'      : team['caps'].mean(),
        'atk_xG_per90'  : atk_xG,
        'atk_xA_per90'  : atk_xA,
    })

team_vectors = pd.DataFrame(rows)
print(f"Built {len(team_vectors)} team vectors")

Built 48 team vectors


In [23]:
team_vectors.to_csv("../data/processed/team_vectors.csv", index=False)

print(f"Teams with xG data: {team_vectors['atk_xG_per90'].notna().sum()} / 48")
print(f"\nNull check:\n{team_vectors.isnull().sum()}")
print(f"\nTop 10 by top11_overall:")
print(team_vectors.nlargest(10, 'top11_overall')[['team', 'top11_overall', 'atk_overall', 'def_overall', 'gk_overall']].to_string())

Teams with xG data: 17 / 48

Null check:
team              0
atk_overall       0
atk_shooting      0
atk_pace          0
atk_dribbling     0
mid_overall       0
mid_passing       0
mid_physic        0
def_overall       0
def_defending     0
def_physic        0
gk_overall        0
top11_overall     0
squad_depth       0
avg_caps          0
atk_xG_per90     31
atk_xA_per90     31
dtype: int64

Top 10 by top11_overall:
           team  top11_overall  atk_overall  def_overall  gk_overall
10       France      86.727273    86.247098    83.232620   80.138075
20        Spain      86.727273    81.815385    81.400000   85.031373
28       Brazil      85.909091    80.678423    80.336463   82.894737
23      Germany      85.272727    80.018750    83.441441   82.685484
40     Portugal      85.272727    80.608696    80.721379   80.768595
8       England      84.909091    83.711111    79.118143   80.468880
30    Argentina      84.636364    81.375221    79.659341   82.073171
39  Netherlands      84.0000

In [24]:
# top 10 by overall squad quality
print("=== TOP 10 BY TOP11 OVERALL ===")
print(team_vectors.nlargest(10, 'top11_overall')[
    ['team', 'top11_overall', 'atk_overall', 'mid_overall', 'def_overall', 'gk_overall', 'squad_depth']
].to_string())

# bottom 5 — should be weak teams
print("\n=== BOTTOM 5 BY TOP11 OVERALL ===")
print(team_vectors.nsmallest(5, 'top11_overall')[
    ['team', 'top11_overall', 'atk_overall', 'def_overall']
].to_string())

# xG coverage
print("\n=== TEAMS WITH XG DATA ===")
print(team_vectors[team_vectors['atk_xG_per90'].notna()][
    ['team', 'atk_xG_per90', 'atk_xA_per90']
].sort_values('atk_xG_per90', ascending=False).to_string())
print(f"\nxG coverage: {team_vectors['atk_xG_per90'].notna().sum()} / 48")

# experience check
print("\n=== TOP 10 BY AVG CAPS (most experienced) ===")
print(team_vectors.nlargest(10, 'avg_caps')[['team', 'avg_caps']].to_string())

# squad depth check — low = deep squad
print("\n=== SQUAD DEPTH (lowest = deepest) ===")
print(team_vectors.nsmallest(10, 'squad_depth')[['team', 'squad_depth', 'top11_overall']].to_string())

# null check
print("\n=== NULL CHECK ===")
print(team_vectors.isnull().sum())

=== TOP 10 BY TOP11 OVERALL ===
           team  top11_overall  atk_overall  mid_overall  def_overall  gk_overall  squad_depth
10       France      86.727273    86.247098    81.769231    83.232620   80.138075     6.193939
20        Spain      86.727273    81.815385    86.390728    81.400000   85.031373     6.393939
28       Brazil      85.909091    80.678423    80.426195    80.336463   82.894737     9.575758
23      Germany      85.272727    80.018750    81.251685    83.441441   82.685484     6.206061
40     Portugal      85.272727    80.608696    84.779528    80.721379   80.768595     6.539394
8       England      84.909091    83.711111    82.788927    79.118143   80.468880     6.175758
30    Argentina      84.636364    81.375221    81.793262    79.659341   82.073171     6.636364
39  Netherlands      84.000000    79.363964    82.137195    81.808576   76.713043     6.000000
29      Belgium      82.909091    80.697723    80.285491    75.920821   80.558333     7.309091
13      Uruguay   

In [25]:
norway = players[players['team'] == 'Norway']
att = norway[norway['broad_pos'] == 'ATT']
print(att[['name', 'understat_xG_per90', 'understat_xA_per90', 'understat_mins']].to_string())

                     name  understat_xG_per90  understat_xA_per90  understat_mins
53      Alexander Sørloth            0.641169            0.056336          1955.0
287        Erling Haaland            0.869950            0.166395          2979.0
522  Jørgen Strand Larsen                 NaN                 NaN             NaN
